# crypto market: practice

This module/project pack is extracted from the local files listed below. Functions and line numbers are source locations; the original program is not executed by practice.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [3_crew/community_contributions/crypto_market/pyproject.toml](../../../../../../3_crew/community_contributions/crypto_market/pyproject.toml)
- [3_crew/community_contributions/crypto_market/README.md](../../../../../../3_crew/community_contributions/crypto_market/README.md)
- [3_crew/community_contributions/crypto_market/src/crypto_market/__init__.py](../../../../../../3_crew/community_contributions/crypto_market/src/crypto_market/__init__.py)
- [3_crew/community_contributions/crypto_market/src/crypto_market/config/agents.yaml](../../../../../../3_crew/community_contributions/crypto_market/src/crypto_market/config/agents.yaml)
- [3_crew/community_contributions/crypto_market/src/crypto_market/config/tasks.yaml](../../../../../../3_crew/community_contributions/crypto_market/src/crypto_market/config/tasks.yaml)
- [3_crew/community_contributions/crypto_market/src/crypto_market/crew.py](../../../../../../3_crew/community_contributions/crypto_market/src/crypto_market/crew.py)
- [3_crew/community_contributions/crypto_market/src/crypto_market/main.py](../../../../../../3_crew/community_contributions/crypto_market/src/crypto_market/main.py)
- [3_crew/community_contributions/crypto_market/src/crypto_market/tools/__init__.py](../../../../../../3_crew/community_contributions/crypto_market/src/crypto_market/tools/__init__.py)
- [3_crew/community_contributions/crypto_market/src/crypto_market/tools/custom_tool.py](../../../../../../3_crew/community_contributions/crypto_market/src/crypto_market/tools/custom_tool.py)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 3. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 4. Evaluate, give feedback, and bound revision

A worker produces a draft and an evaluator judges it against criteria. Feedback becomes input for another attempt. The evaluator here is a small deterministic teaching rule, not a general quality checker.

**Predict:** Is the first draft acceptable? How many attempts will be allowed?

In [ ]:
def evaluate(text):
    return {"accepted": "example:" in text.lower(), "feedback": "Include an example."}
attempts = []
for attempt in range(2):
    draft = "A list stores ordered values." if attempt == 0 else "A list stores ordered values. Example: [1, 2]"
    decision = evaluate(draft)
    attempts.append(decision["accepted"])
    if decision["accepted"]:
        break
print(attempts, draft)
assert attempts == [False, True]

**Check:** The first is rejected and the second accepted. Unlike the Lab 3 one-retry source, this teaching example evaluates the revision too.

**Change one thing:** Make the second draft fail too. Add an explicit message when the finite retry limit is reached.

## 5. Drafting and delivering are separate steps

An agent that formats text is different from a tool that sends email or notifications. This exercise records a delivery intention in memory only.

**Predict:** Will constructing this draft contact anyone?

In [ ]:
outbox = []
draft = {"to": "learner@example.com", "subject": "Practice report", "body": "Fictional findings."}
assert outbox == []
def dry_run_send(message):
    outbox.append({"status": "dry_run", "subject": message["subject"]})
    return {"status": "dry_run"}
result = dry_run_send(draft)
print(result, outbox)
assert len(outbox) == 1

**Check:** No. Neither this draft nor this in-memory dry run sends anything. A provider send/post call is the delivery boundary.

**Change one thing:** Format the body as HTML but keep dry_run_send unchanged. Explain where real delivery would begin.

## 6. Reducer vs. checkpoints vs. conversation identity

A reducer combines state updates. A checkpointer retains state across invocations. A thread_id selects a conversation. This uses SQLite in memory, so it creates no database file.

**Predict:** Does a different thread_id share the same list?

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
class MemoryState(TypedDict):
    turns: Annotated[list[str], add]
builder = StateGraph(MemoryState)
builder.add_node("reply", lambda state: {"turns": ["assistant reply"]})
builder.add_edge(START, "reply")
builder.add_edge("reply", END)
with SqliteSaver.from_conn_string(":memory:") as saver:
    graph = builder.compile(checkpointer=saver)
    config = {"configurable": {"thread_id": "practice-1"}}
    graph.invoke({"turns": ["first user turn"]}, config)
    result = graph.invoke({"turns": ["second user turn"]}, config)
    separate = graph.invoke({"turns": ["new user"]}, {"configurable": {"thread_id": "practice-2"}})
    print(result["turns"], separate["turns"])
    assert len(result["turns"]) == 4 and len(separate["turns"]) == 2

**Check:** No. Same ID continues a thread; a different ID selects another. In-memory SQLite here disappears when the connection closes.

**Change one thing:** Replace :memory: with a deliberately chosen practice database only if you want persistence. Explain when it would survive process exit.

## 7. Roles, tasks, and process

CrewAI separates who does work from what work is required. A sequential process has a fixed task order; a hierarchical process gives a manager coordination responsibility.

**Predict:** Is a role description itself an executable task?

In [ ]:
agents = {"researcher": "Collect fictional evidence", "analyst": "Interpret supplied evidence"}
tasks = [{"name": "research_task", "agent": "researcher", "expected_output": "Findings"},
         {"name": "analysis_task", "agent": "analyst", "expected_output": "Short report"}]
outputs = []
for task in tasks:
    assert task["agent"] in agents
    outputs.append({"task": task["name"], "input_from_previous": list(outputs), "result": task["expected_output"]})
print([item["task"] for item in outputs])
assert len(outputs[1]["input_from_previous"]) == 1

**Check:** No. A task assigns work to a role. This dictionary simulation does not execute CrewAI.

**Change one thing:** Add a reviewer role and task. Describe whether its input should include the analyst output.

## 8. Configuration fills named placeholders

The crew entry point supplies inputs, and YAML descriptions contain placeholders. Configuration names must match the inputs used at runtime.

**Predict:** What happens if motion is not supplied?

In [ ]:
description = "Propose an argument for: {motion}."
inputs = {"motion": "Beginners should practice offline first"}
rendered = description.format(**inputs)
print(rendered)
assert inputs["motion"] in rendered
try:
    description.format(**{})
except KeyError:
    print("Missing placeholder detected")

**Check:** Formatting raises KeyError here. Crew configuration and project environment are separate from the root project.

**Change one thing:** Add a second placeholder named audience and provide its value. Identify matching input keys in the linked source main.py.

## 9. Retrieve evidence before answering

A retrieval workflow selects relevant chunks and passes them as context. This tiny keyword example illustrates the boundary without embeddings, a vector store, or a hosted model.

**Predict:** Which chunk matches the query about Python?

In [ ]:
chunks = [{"id": "a", "text": "Alex writes Python automation."},
          {"id": "b", "text": "Alex enjoys hiking."}]
query = "Python"
evidence = [chunk for chunk in chunks if query.lower() in chunk["text"].lower()]
answer_input = {"question": query, "evidence": evidence}
print(answer_input)
assert [chunk["id"] for chunk in evidence] == ["a"]

**Check:** Chunk a. This keyword selection does not implement the source vector/embedding integrations.

**Change one thing:** Try a query with no matches. Return an explicit lack-of-evidence result instead of inventing an answer.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.